In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

import xgboost as xgb
import shap
df = pd.read_csv("data/insurance_data.csv")
df.head()
df["VehicleAge"] = 2026 - df["RegistrationYear"]

df["Margin"] = df["TotalPremium"] - df["TotalClaims"]

df["HasClaim"] = (df["TotalClaims"] > 0).astype(int)
df = df.dropna()
cat_cols = df.select_dtypes(include=["object"]).columns

le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))
    df_sev = df[df["HasClaim"] == 1]

X = df_sev.drop(["TotalClaims", "HasClaim"], axis=1)
y = df_sev["TotalClaims"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
lr = LinearRegression()
lr.fit(X_train, y_train)

pred_lr = lr.predict(X_test)

print("LR RMSE:", np.sqrt(mean_squared_error(y_test, pred_lr)))
print("LR R2:", r2_score(y_test, pred_lr))
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)

print("RF RMSE:", np.sqrt(mean_squared_error(y_test, pred_rf)))
print("RF R2:", r2_score(y_test, pred_rf))
xg = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1)
xg.fit(X_train, y_train)

pred_xg = xg.predict(X_test)

print("XGB RMSE:", np.sqrt(mean_squared_error(y_test, pred_xg)))
print("XGB R2:", r2_score(y_test, pred_xg))
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "XGBoost"],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, pred_lr)),
        np.sqrt(mean_squared_error(y_test, pred_rf)),
        np.sqrt(mean_squared_error(y_test, pred_xg))
    ],
    "R2": [
        r2_score(y_test, pred_lr),
        r2_score(y_test, pred_rf),
        r2_score(y_test, pred_xg)
    ]
})

results
explainer = shap.Explainer(xg)
shap_values = explainer(X_test)

shap.summary_plot(shap_values, X_test)
X = df.drop(["HasClaim", "TotalClaims"], axis=1)
y = df["HasClaim"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

clf = xgb.XGBClassifier()
clf.fit(X_train, y_train)

probs = clf.predict_proba(X_test)[:,1]
df["Pred_Prob"] = clf.predict_proba(X)[:,1]
df["Pred_Severity"] = xg.predict(X)

df["Pred_Premium"] = (df["Pred_Prob"] * df["Pred_Severity"]) * 1.2

